# Multi-Agent Automatic Code Review System - Enhanced Documentation

## Introduction  

This system implements a `multi-agent system using CrewAI framework` to automatically review code changes. The system decompose code review tasks across specialized AI agents —a Senior Developer, Security Engineer, and Tech Lead—each-  with distinct expertise and responsibilities.   
This approach leverages the strengths of multi-agent collaboration to provide comprehensive, efficient, and nuanced code quality assessments that would traditionally require human expertise. Thus, this strategy uses a development team in order to automate initial code reviews, saving in this way  valuable time.  

It evaluates:
- Code quality
- Security risks
- Final approval decisions


## Understanding the context  

### The Problem

As a technical leader, you understand that many code issues follow recurring patterns—such as style inconsistencies, common bugs, and known vulnerabilities—that can be automatically detected and evaluated. However, not all code changes are straightforward; some require deeper contextual understanding and human judgment. This creates a challenge: how to efficiently scale code review processes while maintaining high standards of quality and security. An effective solution must be able to analyze code changes, identify potential issues, and autonomously decide whether to approve changes, suggest fixes, or escalate to a human reviewer.

### The Solution : code review code 

To address this challenge, a multi-agent system is introduced to automate the code review process. The system is composed of specialized agents—such as a Senior Developer, Security Engineer, and Tech Lead—each responsible for a specific aspect of the review workflow. These agents operate sequentially, performing quality analysis, security assessment, and final decision-making. By mimicking real-world engineering roles, the system ensures that code is evaluated from multiple perspectives, resulting in more reliable and structured outcomes. The final output provides a clear decision: approve, request changes, or escalate for human review.

### Why CrewAI? 

CrewAI is particularly well-suited for implementing this solution because it enables the creation of role-based agents with clearly defined goals, backstories, and toolsets. It provides built-in support for task orchestration, allowing agents to collaborate in a structured and sequential workflow. Additionally, CrewAI facilitates seamless integration of external tools—such as search and scraping utilities—enhancing the system’s ability to perform real-world analysis. This makes it an ideal framework for building scalable, modular, and extensible multi-agent systems that closely resemble real engineering workflows.


## System Architecture  

## The three agents team 

###  **Senior Developer Agent**
**Role:** Code Quality Expert  
**Goal:** Evaluate code quality and determine required fixes  
**Backstory:** Expert in maintainability, readability, and pragmatic engineering decisions  
**Tools:** None  


### **Security Engineer Agent**
**Role:** Security Analysis Specialist  
**Goal:** Identify vulnerabilities and assess risk levels  
**Backstory:** متخصص in real-world attack vectors and risk prioritization  

**Tools:**
- SerperDevTool → Search vulnerabilities & CVEs  
- ScrapeWebsiteTool → Extract detailed security insights  

### **Tech Lead Agent**
**Role:** Review Orchestrator  
**Goal:** Decide approval path for code changes  
**Backstory:** Balances quality, security, and delivery timelines  
**Tools:** None  


### Code Review Crew — Sequential Workflow (Task-Oriented + Tools)

```bash  
  
Code Changes / Pull Request
   ↓
[👨‍💻 Senior Developer Agent]  
   Role: Code Quality Expert  
   Goal: Evaluate code quality and determine which issues must be fixed  
   Backstory: A highly experienced developer known for enforcing best practices, 
   making pragmatic decisions on code quality, and balancing perfection with delivery speed  
   Tools: None (code analysis + reasoning)

   ↓
   [Quality Analysis Task]
   ← Evaluates code style, readability, and consistency  
   ← Detects bugs, edge cases, and anti-patterns  
   ← Assesses maintainability and technical debt  
   ← Decides which issues must be fixed before approval  
   ↓

[🔐 Security Engineer Agent]  
   Role: Security Analysis Specialist  
   Goal: Identify vulnerabilities and assess their risk level  
   Backstory: A security-focused engineer with deep expertise in identifying 
   real-world attack vectors and prioritizing risks based on impact and likelihood  

   Tools:
   - SerperDevTool (search for known vulnerabilities, CVEs, best practices)  
   - ScrapeWebsiteTool (extract detailed security information from sources)  

   ↓
   [Security Review Task]
   ← Scans code for vulnerabilities (e.g., injection, auth flaws, secrets exposure)  
   ← Uses SerperDevTool to check known vulnerabilities and exploit patterns  
   ← Uses ScrapeWebsiteTool to extract detailed security insights from sources  
   ← Assigns risk levels (low / medium / high / critical)  
   ← Determines if issues should block approval  
   ↓

[🧑‍🏫 Tech Lead Agent]  
   Role: Review Orchestrator  
   Goal: Determine approval path for code changes  
   Backstory: A seasoned technical leader responsible for ensuring code quality, 
   security, and efficient review workflows across the team  
   Tools: None (decision-making + coordination)

   ↓
   [Review Decision Task]
   ← Analyzes quality and security findings  
   ← Decides approval outcome:
        • ✅ Auto-approve (no critical issues)  
        • 🔧 Request changes (fix required issues)  
        • 👀 Escalate to human review (high risk or uncertainty)  
   ↓

📦 Final Outcome
   - Approved  
   - Changes Required  
   - Escalated for Manual Review  
``` 

## Installation & Setup 

Firstly, install required libraries (see `requirements.txt`), then import the necessary modules and configure API Keys.  

In [50]:
%pip install -r requirements.txt

In [ ]:
# Import Required Modules

import os
import dill
import unittests
import warnings
import inspect

from crewai import Agent, Task, Crew
from google.colab import userdata

print("unittests module imported successfully!")

# Suppress warnings
warnings.filterwarnings('ignore')

unittests module imported successfully!


Inspect the `unitests.py` file

In [ ]:
#import inspect
#import os

# Get the file path of the unittests module
unittests_file_path = inspect.getfile(unittests)
print(f"Path to unittests.py: {unittests_file_path}")

# Read and print the content of the file
with open(unittests_file_path, 'r') as f:
    content = f.read()
print("\n--- Content of unittests.py ---\n")
print(content)

Path to unittests.py: /content/unittests.py

--- Content of unittests.py ---

import sys
import itertools

from crewai import Agent, Task, Crew, LLM, TaskOutput
from crewai_tools import WebsiteSearchTool
from pydantic import BaseModel

from dlai_grader.grading import test_case
from typing import Tuple, Any
import json

def print_results(cases):
    failed_cases = [t for t in cases if t.failed]
    if len(failed_cases) == 0:
        print("\033 All tests passed!\n")
    else:
        print(f"\033 You have {len(failed_cases)} failed tests:\n")
        for failed_case in failed_cases:
            feedback_msg = ""
            feedback_msg += f"Failed test case: {failed_case.msg}. \nExpected:\n{failed_case.want},\nbut got:\n{failed_case.got}.\n\n"
            print(feedback_msg)


def test_tools(serper_search_tool, scrape_website_tool):
    cases = []

    # Test 1: Check if the serper_search_tool can run
    t = test_case()
    try:
        content = serper_search_tool.run(search_query="S

Next, set up the environment variables to connect to the APIs, and  create the LLM instance you will use for your Agents

In [53]:
#import os
os.environ["MODEL"] = "gemini-2.5-flash"   #replace with the model you want

In [54]:
#  Secure API key setup
def setup_api_key(key_name: str) -> str:
    """Retrieve and validate API key from Colab secrets."""
    try:
        api_key = userdata.get(key_name)
        if not api_key:
            raise ValueError(f"{key_name} not found in Colab secrets")

        # Display masked key
        masked = api_key[:4] + "****" + api_key[-4:] if len(api_key) > 8 else "***"
        print(f"✓ {key_name} loaded successfully ({masked})")
        return api_key
    except Exception as e:
        print(f"❌ Error loading {key_name}: {e}")
        raise

gemini_api_key = setup_api_key('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = gemini_api_key

✓ GEMINI_API_KEY loaded successfully (AIza****AWwQ)


## Enhanced Code Quality Issues in the give file.txt    

Let's review the `pull request (PR)` with the code changes from the `code_changes.txt`.

In [55]:
# read and save the policies content
with open('code_changes.txt', 'r') as file:
    code_changes = file.read()
print(code_changes)

diff --git a/app/user_auth.py b/app/user_auth.py
index 8f23c4d..b9e7f2a 100644
--- a/app/user_auth.py
+++ b/app/user_auth.py
@@ -1,7 +1,32 @@
+from datetime import datetime
+import time
+
 def authenticate_user(username, password):
+    # Check if username or password is empty
+    if not username or not password:
+        return False
+    
+    # Query the database for the user
     user = db.query(f"SELECT * FROM users WHERE username = '{username}'")
+    
+    # Verify the user exists and password matches
     if user and user.password == password:
+        # Set session variables
         session['user_id'] = user.id
+        session['login_time'] = datetime.now()
+        
+        # Update last login timestamp
+        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")
+        
+        print(f"User {username} logged in successfully")
         return True
-    return False
+    else:
+        # Sleep to prevent timing attacks
+        time.sleep(1)
+       

**Explanation outpput** : Based on the above ouptut, the following several `Security Vulnerabilities` were identified: 

```python
# Issue 1: SQL INJECTION in user_auth.py
❌ user = db.query(f"SELECT * FROM users WHERE username = '{username}'")
✅ user = db.query("SELECT * FROM users WHERE username = ?", (username,))

# Issue 2: Plaintext Password Comparison
❌ if user and user.password == password:
✅ if user and verify_password(password, user.password_hash):

# Issue 3: SQL INJECTION in UPDATE statement
❌ db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")
✅ db.execute("UPDATE users SET last_login = NOW() WHERE id = ?", (user.id,))
```


## Define the agents and tasks 

### Agent Responsibilities

| Agent | Role | Key Tasks | Tools |
|-------|------|-----------|-------|
| **Senior Developer** | Code Quality Evaluator | Analyze code style, bugs, maintainability | None (uses context only) |
| **Security Engineer** | Vulnerability Analyst | Identify security risks, compliance issues | SerperDevTool, ScrapeWebsiteTool |
| **Tech Lead** | Decision Coordinator | Synthesize reviews, make final recommendations | Context from other agents |

### 🛠️ Tools

| Tool               | Used By                | Purpose                                  |
|--------------------|------------------------|------------------------------------------|
| SerperDevTool      | Security Engineer      | Search vulnerabilities & CVEs             |
| ScrapeWebsiteTool  | Security Engineer      | Extract detailed security information     |
| None               | Planner, Tech Lead     | Reasoning & decision-making               |


In [56]:
# import the required tools
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
# get the Serper API key
#from google.colab import userdata
#serper_api_key = userdata.get('SERPER_API_KEY')
#import os
#os.environ["SERPER_API_KEY"] = serper_api_key

In [57]:
# Get the SERPER api key securely

def setup_api_key(key_name: str) -> str:
    """Retrieve and validate API key from Colab secrets."""
    try:
        api_key = userdata.get(key_name)
        if not api_key:
            raise ValueError(f"{key_name} not found in Colab secrets")

        # Display masked key
        masked = api_key[:4] + "****" + api_key[-4:] if len(api_key) > 8 else "***"
        print(f"✓ {key_name} loaded successfully ({masked})")
        return api_key
    except Exception as e:
        print(f"❌ Error loading {key_name}: {e}")
        raise

serper_api_key = setup_api_key('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serper_api_key

✓ SERPER_API_KEY loaded successfully (c783****5a4c)


In [ ]:
# ===== TOOL CONFIGURATION =====
# Configure SerperDevTool for security best practices search
# Limited to OWASP to ensure security-focused results

serper_search_tool = SerperDevTool(search_url="https://owasp.org", base_url="https://google.serper.dev/search")

# Configure ScrapeWebsiteTool for content extraction
# Used to fetch detailed security documentation
scrape_website_tool = ScrapeWebsiteTool()


In [59]:
# test the tools
unittests.test_tools(serper_search_tool, scrape_website_tool)

 All tests passed!



**Explanation output** The message `All tests passed` means that every test case defined within the unittests.py module for the `serper_search_tool` and `srape_website_tool` executed successfully without encountering any errors or unexpected behaviors. In simpler terms, the tools are working as expected according to the defined tests.

In [60]:
# Path to the unittests.py file
unittests_file_path = '/content/unittests.py'

# Read the content of the file
with open(unittests_file_path, 'r') as f:
    content = f.read()

# Replace the incorrect argument name
old_string = 'content = serper_search_tool.run(query="SQL issues")'
new_string = 'content = serper_search_tool.run(search_query="SQL issues")'
modified_content = content.replace(old_string, new_string)

# Write the modified content back to the file
with open(unittests_file_path, 'w') as f:
    f.write(modified_content)

print(f"Modified {unittests_file_path} to fix SerperDevTool argument name.")

Modified /content/unittests.py to fix SerperDevTool argument name.


## Define the agents  

Now it is time to define each agent. As it can be seen in the above workflow, each agent is characterised by a specific 'role', 
`goal`and `backstory`. 

In [ ]:
# ===== AGENT DEFINITION =====
# Role: Describes the agent's job title and primary function
# Goal: The specific outcome the agent should achieve
# Backstory: Provides context about the agent's expertise (helps LLM roleplay)
# LLM: The language model powering the agent's reasoning
# Verbose: Shows detailed execution steps (useful for debugging)


# Create the senior developer agent
senior_developer = Agent(
    role="Senior Developer",  
    goal="Evaluate code changes for style, potential bugs, and maintainability issues, determining which issues are critical and must be addressed before code approval.",
    backstory=(
        "As a Senior Developer, I specialize in maintaining high code quality standards. "
        "I meticulously review pull requests to identify code smells, structural issues, and potential bugs. "
        "My expertise lies in differentiating between critical flaws that block deployment and minor improvements that can be addressed later, ensuring robust and maintainable codebases."
    ),
    verbose=True,  # set verbose (suggested: True)
)

In [62]:
# test the senior developer agent
unittests.test_senior_developer_agent(senior_developer)

 All tests passed!



In [63]:
# Create the security engineer agent
security_engineer = Agent(
    #llm=llm, # Pass the explicitly created LLM instance here
    role="Security Engineer",
    goal="Identify and assess security vulnerabilities in code changes, determine their risk levels, and decide if they block approval of the pull request.",
    backstory=(
        "As a Security Engineer, I possess deep expertise in application security, "
        "penetration testing, and vulnerability assessment. "
        "I meticulously review code for potential exploits, weaknesses, and compliance issues, "
        "providing clear recommendations and making critical decisions on whether security flaws "
        "warrant blocking a code release or can be mitigated post-release."
    ),
    verbose=True,  # set verbose (suggested: True)
    tools=[serper_search_tool, scrape_website_tool],  # assign the tools you created
)

In [64]:
# test the security engineer agent
unittests.test_security_engineer_agent(security_engineer)

 All tests passed!



In [65]:
# Create the tech lead agent
tech_lead = Agent(
    #llm=llm,
    role="Tech Lead",
    goal="Determine approval paths for code changes based on quality and security reviews, making final decisions on approval, required fixes, or escalation to human review.",
    backstory=(
            "As a Tech Lead, I oversee the entire code review process. My expertise lies in synthesizing feedback from both development and security teams to make informed decisions. I am responsible for ensuring that all code changes meet our quality and security standards, and for streamlining the approval workflow, escalating to human review only when absolutely necessary."
        ),
    verbose=True,  # set verbose (suggested: True)
)

In [66]:
# test the tech lead agent
unittests.test_tech_lead_agent(tech_lead)

 All tests passed!



## Define the Tasks for each Agent  

After having set up the agents, it is necessary to define the tasks each of them will perform. In particular three tasks (one for each agent):  

- **Quality Analysis Task**: Evaluate code changes for style, bugs, and maintainability, deciding which issues must be fixed before approval.  
     
- **Security Review Task**: Examine code for security vulnerabilities, determining risk levels and whether security issues should block approval.  
     
- **Review Decision Task**: Analyze the quality and security findings to decide if changes can be automatically approved, need specific fixes, or require human review.  

#### General guidelines for creating Tasks:
When creating each task, you'll need to define these key parameters:

- `description`: A clear explanation of what the task involves
- `expected_output`: The format and content the task should produce
- `agent`: Which agent will perform this task
- `context` (optional): Define what tasks' output, including multiple, should be used as context for another task. You can learn more about context in the [docs](https://docs.crewai.com/en/concepts/tasks#referring-to-other-tasks). In this case, you will only need to set the context for the last task.

### Create Quality Analysis Task  

Creation a **Quality Analysis task** for code quality evaluation by:
- Writing a `description` with steps instructing the agent to review code, identify potential bugs or issues, and decide if the issues are critical or minor.
    - The task should read the code changes from the provided `code_changes`.
    - Use `{code_changes}` in your `description`, but do NOT use an f string.
- Specifying that the `expected_output` should be a `JSON` with exactly these keys:
    - `critical_issues`: array of issues that must be fixed
    - `minor_issues`: array of suggested improvements
    - `reasoning`: explanation of decisions
- Assigning the task to the **Senior Developer** agent.

In [67]:
# Create the quality analysis task
analyze_code_quality = Task(
    description=(
        "Review the provided code changes: {code_changes}. "
        "Identify any stylistic issues, potential bugs, or maintainability concerns. "
        "Determine which issues are critical and must be fixed before approval, and which are minor suggestions for improvement."
    ),

    expected_output=(
        "A JSON object with the following keys: "
        "'critical_issues' (array of issues that must be fixed), "
        "'minor_issues' (array of suggested improvements), and "
        "'reasoning' (explanation of decisions)."
    ),

    name="Analyze Code Quality",
    agent=senior_developer
)


In [68]:
# test the quality analysis task
unittests.test_analyze_code_quality_task(analyze_code_quality)

 All tests passed!



### Create Security Review Task  

Creation a **Security Review task** for security evaluation by:
- Completing the `description` with steps instructing the agent to examine code for vulnerabilities, identify security issues, determine risk levels, and decide if issues should block approval.
    - The task should read the code changes from the provided `code_changes`.
    - Use `{code_changes}` in your `description`, but do NOT use an f string.
- Specifying that the `expected_output` should be a `JSON` with exactly these keys:
    - `security_vulnerabilities`: array of identified issues with risk levels
    - `blocking`: boolean indicating if security issues should block approval
    - `highest_risk`: the most severe risk level found
    - `security_recommendations`: specific fixes for vulnerabilities
- Assigning the task to the **Security Engineer** agent.

In [69]:
# Create the security review task
review_security = Task(
    description=(
        "Review the provided code changes: {code_changes}. "
        "Examine the code for security vulnerabilities, identify security issues, determine their risk levels (e.g., Critical, High, Medium, Low), and decide if these issues should block approval of the pull request. "
        "Use the SerperDevTool to find the most relevant security best practices from OWASP "
        "and pass the URLs to the ScrapeWebsiteTool to get detailed information."
    ),

    expected_output=(
        "A JSON object with the following keys: "
        "'security_vulnerabilities' (array of identified issues with risk levels), "
        "'blocking' (boolean indicating if security issues should block approval), "
        "'highest_risk' (the most severe risk level found), and "
        "'security_recommendations' (specific fixes for vulnerabilities)."
    ),

    agent=security_engineer,
    name="Review Security",
)

In [70]:
# test the review security task
unittests.test_review_security_task(review_security)

 All tests passed!




### Create Review Decision Task  

Creation a **Review Decision task** for review coordination by:
- Writing a `description` with steps instructing the agent to determine if the PR can be approved, decide on next steps, explain the decision.
    - The task should read the code changes from the provided `code_changes`.
    - Use `{code_changes}` in your `description`, but do NOT use an f string.
- Specifying that the `expected_output` should be a short report that includes the final decision, required changes (if any), approval comments (if approving), escalation reasoning (if escalating), and additional recommendations.
- Assigning the task to the **Tech Lead** agent.
- Assigning `context` to the agent. The Review Decision task needs the output of both of the previous tasks to make the decision, so you need to pass both tasks as context.

In [71]:
# Create the review decision task
make_review_decision = Task(
    description=(
        "Analyze the code changes: {code_changes}, along with the quality analysis and security review results. "
        "Determine if the pull request can be automatically approved, requires specific fixes, or needs escalation for human review. "
        "Explain the decision thoroughly."
    ),

    expected_output=(
        "A short report including: "
        "'final_decision' (Approved, Fixes Required, or Escalated), "
        "'required_changes' (list of specific fixes, if any), "
        "'approval_comments' (if approved), "
        "'escalation_reasoning' (if escalated), and "
        "'additional_recommendations'."
    ),

    agent=tech_lead,
    context=[analyze_code_quality, review_security], # add the two previous tasks as context
    name="Review Decision"
)

In [72]:
# test the review decision task
unittests.test_make_review_decision_task(make_review_decision)

 All tests passed!



##  Define and kick off the Crew

###  Create the Crew  

In [ ]:
# Create crew with all agents and tasks
crew = Crew(
    agents=[senior_developer, security_engineer, tech_lead],   # add the list of agents
    tasks=[analyze_code_quality, review_security, make_review_decision], # add the list of tasks
)

In [74]:
# test the crew
unittests.test_crew(crew)

 All tests passed!



Next, define all the inputs to kickoff the crew. Run the cell below to define the `inputs` dictionary with the `code_changes`, which then will be passed as context to the Crew.

In [75]:
# define the inputs dictionary for the crew
inputs = {
    "code_changes": code_changes,
}

### Kickoff the Crew  

In [ ]:
# ===== CREW EXECUTION =====
#print("=" * 60)
#print("INITIATING MULTI-AGENT CODE REVIEW")
#print("=" * 60)

# Create crew with all agents and tasks
#code_review_crew = Crew(
#    agents=[senior_developer, security_engineer, tech_lead],
#    tasks=[analyze_code_quality, review_security, make_review_decision],
#    verbose=True,
#    memory=True
#)

# Execute the crew
#print(f"\n📋 Reviewing code changes:\n{code_changes}\n")
#result = code_review_crew.kickoff(inputs={"code_changes": code_changes})

#print("\n" + "=" * 60)
#print("REVIEW COMPLETE")
#print("=" * 60)
#print(f"\nFinal Decision:\n{result}")

In [ ]:
# Execute the crew
result = crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Task: Review the provided code changes: diff --git a/app/user_auth.py b/app/user_auth.py                       │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                                                                                     │
│  +        # Sleep to prevent timing attacks                                                                     │
│  +        time.sleep(1)                                

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Developer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "critical_issues": [                                                                                         │
│      "**SQL Injection Vulnerability**: The line `user = db.query(f\"SELECT * FROM users WHERE username =        │
│  '{username}'\")` is highly vulnerable to SQL injection. User-provided `username` is directly concatenated      │
│  into the SQL query string without proper sanitization or parameterization. An attacker could easily inject     │
│  malicious SQL to bypass authentication or access/modify data.",                                                │
│      "**Insecure Password Storage and Verification**: The comparison `user.password == password` strongly       │
│  suggests that passwords are being stored in plaintext or a reversibly encrypted format in the database. This   │
│  is a critical security flaw. Passwords must always be hashed using a strong, one-way algorithm (e.g., bcrypt,  │
│  Argon2) with a unique salt for each user. Verification should involve hashing the user-provided password and   │
│  comparing the resulting hash with the stored hash.",                                                           │
│      "**Syntax Error in `logout_user`**: The line `return True.` contains a syntax error (the trailing dot).    │
│  It should be `return True`. This will cause the function to fail at runtime."                                  │
│    ],                                                                                                           │
│    "minor_issues": [                                                                                            │
│      "**Global Dependencies**: The `authenticate_user` function directly accesses `db` and `session` objects,   │
│  implying they are global or implicitly imported. This makes the function harder to test in isolation and       │
│  reduces its reusability. Passing these as arguments (dependency injection) or using a clear dependency         │
│  management pattern would improve maintainability.",                                                            │
│      "**Logging Practices**: Using `print()` statements for logging (`print(f\"User {username} logged in        │
│  successfully\")`) is generally not suitable for production applications. A proper logging framework (e.g.,     │
│  Python's `logging` module) should be used to control log levels, destinations (files, syslog, stdout/stderr),  │
│  and formatting, especially for sensitive information like usernames.",                                         │
│      "**Database Timestamp Function Portability**: The `NOW()` function used in `db.execute(f\"UPDATE users     │
│  SET last_login = NOW() WHERE id = {user.id}\")` is specific to certain SQL dialects (e.g., MySQL,              │
│  PostgreSQL). For better database portability, `CURRENT_TIMESTAMP` is a more standard SQL function, or it       │
│  might be better to let the database driver or ORM handle current timestamps.",                                 │
│      "**Incomplete Timing Attack Mitigation**: While `time.sleep(1)` on failed login is a good attempt to       │
│  mitigate timing attacks, it's not a complete solution.

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Task: Review the provided code changes: diff --git a/app/user_auth.py b/app/user_auth.py                       │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                                                                                     │
│  +        # Sleep to prevent timing attacks                                                                     │
│  +        time.sleep(1)                                

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'OWASP SQL Injection prevention', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'SQL Injection Prevention - OWASP Cheat Sheet Series', 'link': 'htt...
Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'OWASP insecure authentication prevention', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Insecure Authentication Methods and Default Credentials'...
Tool read_website_content executed with result: The following text is scraped website content:
SQL Injection Prevention - OWASP Cheat Sheet Series
Skip to content
OWASP Cheat Sheet Series
SQL Injection Prevention
Initializing search
OWASP/CheatShee...
Tool read_website_content executed with result: The following text is scraped website content:
SQL Injection | OWASP Foundation
For full functionality of this site it is necessary to enable JavaScript. Here are the instruc

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Security Engineer                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "security_vulnerabilities": [                                                                                │
│      {                                                                                                          │
│        "issue": "SQL Injection Vulnerability",                                                                  │
│        "risk_level": "Critical",                                                                                │
│        "description": "The line `user = db.query(f\"SELECT * FROM users WHERE username = '{username}'\")` and   │
│  `db.execute(f\"UPDATE users SET last_login = NOW() WHERE id = {user.id}\")` are highly vulnerable to SQL       │
│  injection. User-provided `username` is directly concatenated into the SQL query string without proper          │
│  sanitization or parameterization. An attacker could easily inject malicious SQL to bypass authentication or    │
│  access/modify data."                                                                                           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Insecure Password Storage and Verification",                                                   │
│        "risk_level": "Critical",                                                                                │
│        "description": "The comparison `user.password == password` strongly suggests that passwords are being    │
│  stored in plaintext or a reversibly encrypted format in the database. This is a critical security flaw.        │
│  Passwords must always be hashed using a strong, one-way algorithm (e.g., bcrypt, Argon2) with a unique salt    │
│  for each user. Verification should involve hashing the user-provided password and comparing the resulting      │
│  hash with the stored hash."                                                                                    │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Syntax Error in `logout_user`",                                                                │
│        "risk_level": "Critical",                                                                                │
│        "description": "The line `return True.` contains a syntax error (the trailing dot). It should be         │
│  `return True`. This will cause the function to fail at runtime, preventing users from logging out properly."   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "issue": "Incomplete Timing Attack Mitigation",                                                          │
│        "risk_level": "Medium",                         

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Task: Analyze the code changes: diff --git a/app/user_auth.py b/app/user_auth.py                               │
│  index 8f23c4d..b9e7f2a 100644                                                                                  │
│  --- a/app/user_auth.py                                                                                         │
│  +++ b/app/user_auth.py                                                                                         │
│  @@ -1,7 +1,32 @@                                                                                               │
│  +from datetime import datetime                                                                                 │
│  +import time                                                                                                   │
│  +                                                                                                              │
│   def authenticate_user(username, password):                                                                    │
│  +    # Check if username or password is empty                                                                  │
│  +    if not username or not password:                                                                          │
│  +        return False                                                                                          │
│  +                                                                                                              │
│  +    # Query the database for the user                                                                         │
│       user = db.query(f"SELECT * FROM users WHERE username = '{username}'")                                     │
│  +                                                                                                              │
│  +    # Verify the user exists and password matches                                                             │
│       if user and user.password == password:                                                                    │
│  +        # Set session variables                                                                               │
│           session['user_id'] = user.id                                                                          │
│  +        session['login_time'] = datetime.now()                                                                │
│  +                                                                                                              │
│  +        # Update last login timestamp                                                                         │
│  +        db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")                               │
│  +                                                                                                              │
│  +        print(f"User {username} logged in successfully")                                                      │
│           return True                                                                                           │
│  -    return False                                                                                              │
│  +    else:                                                                                                     │
│  +        # Sleep to prevent timing attacks                                                                     │
│  +        time.sleep(1)                                

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Lead                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **final_decision**: Fixes Required                                                                             │
│                                                                                                                 │
│  **required_changes**:                                                                                          │
│  1.  **SQL Injection Prevention**:                                                                              │
│      *   Modify `user = db.query(f"SELECT * FROM users WHERE username = '{username}'")` to use parameterized    │
│  queries. The `username` must be passed as a parameter to the database driver's query method, not directly      │
│  concatenated into the SQL string.                                                                              │
│      *   Modify `db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")` to use parameterized  │
│  queries for `user.id`. Although `user.id` is internally derived, using parameters consistently is a best       │
│  practice.                                                                                                      │
│  2.  **Secure Password Storage and Verification**:                                                              │
│      *   Implement a strong, one-way password hashing algorithm (e.g., `bcrypt`, `Argon2`) with a unique,       │
│  randomly generated salt for each user.                                                                         │
│      *   The database should store the password hash and salt, not the plaintext password.                      │
│      *   Modify the authentication logic to hash the provided `password` with the user's stored salt and then   │
│  compare this new hash to the stored hash using a constant-time comparison function. The comparison             │
│  `user.password == password` must be removed.                                                                   │
│  3.  **Fix Syntax Error**: Correct the `logout_user` function from `return True.` to `return True`.             │
│  4.  **Improve Timing Attack Mitigation**: Ensure that password hashing and comparison operations are           │
│  performed in constant time, beyond just adding a `time.sleep(1)`. This typically involves using a              │
│  cryptographically secure hashing library that inherently uses constant-time comparison (if applicable) or      │
│  implementing a constant-time comparison function for the hash strings.                                         │
│                                                                                                                 │
│  **approval_comments**: N/A                                                                                     │
│                                                                                                                 │
│  **escalation_reasoning**: N/A (The issues are clearly defined and actionable, requiring direct code fixes      │
│  rather than immediate escalation for a broader architectural or policy review.)                                │
│                                                                                                                 │
│  **additional_recommendations**:                                                                                │
│  *   **Refactor Global Dependencies**: Consider passing

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



In [ ]:
# Print the final report 
from IPython.display import Markdown
Markdown(result.tasks_output[2].raw)

**final_decision**: Fixes Required

**required_changes**:
1.  **SQL Injection Prevention**:
    *   Modify `user = db.query(f"SELECT * FROM users WHERE username = '{username}'")` to use parameterized queries. The `username` must be passed as a parameter to the database driver's query method, not directly concatenated into the SQL string.
    *   Modify `db.execute(f"UPDATE users SET last_login = NOW() WHERE id = {user.id}")` to use parameterized queries for `user.id`. Although `user.id` is internally derived, using parameters consistently is a best practice.
2.  **Secure Password Storage and Verification**:
    *   Implement a strong, one-way password hashing algorithm (e.g., `bcrypt`, `Argon2`) with a unique, randomly generated salt for each user.
    *   The database should store the password hash and salt, not the plaintext password.
    *   Modify the authentication logic to hash the provided `password` with the user's stored salt and then compare this new hash to the stored hash using a constant-time comparison function. The comparison `user.password == password` must be removed.
3.  **Fix Syntax Error**: Correct the `logout_user` function from `return True.` to `return True`.
4.  **Improve Timing Attack Mitigation**: Ensure that password hashing and comparison operations are performed in constant time, beyond just adding a `time.sleep(1)`. This typically involves using a cryptographically secure hashing library that inherently uses constant-time comparison (if applicable) or implementing a constant-time comparison function for the hash strings.

**approval_comments**: N/A

**escalation_reasoning**: N/A (The issues are clearly defined and actionable, requiring direct code fixes rather than immediate escalation for a broader architectural or policy review.)

**additional_recommendations**:
*   **Refactor Global Dependencies**: Consider passing `db` and `session` objects into `authenticate_user` as arguments (dependency injection) rather than relying on global access. This improves testability and modularity.
*   **Implement Robust Logging**: Replace `print()` statements with a dedicated logging framework (e.g., Python's `logging` module) for production-grade logging. This allows for controlled log levels, output destinations, and better handling of sensitive information.
*   **Enhance Database Portability for Timestamps**: For better database portability, use `CURRENT_TIMESTAMP` instead of `NOW()` in SQL queries, or rely on an ORM/database driver to handle current timestamp generation.
*   **Consider Rate Limiting**: Implement a more robust rate-limiting mechanism for login attempts to further mitigate brute-force and timing attacks, complementing the improved password hashing and constant-time comparisons.

In [78]:
# Save the results
with open("results.dill", "wb") as f:
    dill.dump(result, f)

In [79]:
# Get the raw output of the first task (Analyze Code Quality)
raw_output_task_0 = result.tasks_output[0].raw
print("--- Raw Output of Analyze Code Quality Task ---")
print(raw_output_task_0)


--- Raw Output of Analyze Code Quality Task ---
```json
{
  "critical_issues": [
    "**SQL Injection Vulnerability**: The line `user = db.query(f\"SELECT * FROM users WHERE username = '{username}'\")` is highly vulnerable to SQL injection. User-provided `username` is directly concatenated into the SQL query string without proper sanitization or parameterization. An attacker could easily inject malicious SQL to bypass authentication or access/modify data.",
    "**Insecure Password Storage and Verification**: The comparison `user.password == password` strongly suggests that passwords are being stored in plaintext or a reversibly encrypted format in the database. This is a critical security flaw. Passwords must always be hashed using a strong, one-way algorithm (e.g., bcrypt, Argon2) with a unique salt for each user. Verification should involve hashing the user-provided password and comparing the resulting hash with the stored hash.",
    "**Syntax Error in `logout_user`**: The line `re

In [80]:
import json

# Get the raw output of the first task (Analyze Code Quality)
raw_output_task_0 = result.tasks_output[0].raw

# Extract the JSON string by removing the markdown code block delimiters
json_string = raw_output_task_0.strip().replace('```json', '').replace('```', '')

try:
    # Parse the cleaned string as a JSON dictionary
    parsed_output = json.loads(json_string)
    print("✅ Can be parsed as JSON dictionary")
    print(f"Keys: {list(parsed_output.keys())}")
except json.JSONDecodeError as e:
    print(f"❌ Cannot parse as JSON: {e}")
    print("Raw string content for debugging:")
    print(json_string)

✅ Can be parsed as JSON dictionary
Keys: ['critical_issues', 'minor_issues', 'reasoning']


**Explanation output** The Analyze Code Quality task's output has been successfully parsed as a JSON dictionary, confirming that the LLM is indeed producing the correct structure, even with the markdown wrapping.

Now check the second task (Review Security).

In [ ]:
# check the result of the first task

# Get the raw output
#raw_output = result.tasks_output[1].raw

# See if it can be parsed as a dictionary, and get the keys
#get_dict_keys(raw_output)

  ❌ Cannot parse as JSON



In [82]:
# Get the raw output of the first task (Analyze Code Quality)
raw_output_task_1 = result.tasks_output[1].raw
print("--- Raw Output of Review Securiry Task ---")
print(raw_output_task_1)


--- Raw Output of Review Securiry Task ---
```json
{
  "security_vulnerabilities": [
    {
      "issue": "SQL Injection Vulnerability",
      "risk_level": "Critical",
      "description": "The line `user = db.query(f\"SELECT * FROM users WHERE username = '{username}'\")` and `db.execute(f\"UPDATE users SET last_login = NOW() WHERE id = {user.id}\")` are highly vulnerable to SQL injection. User-provided `username` is directly concatenated into the SQL query string without proper sanitization or parameterization. An attacker could easily inject malicious SQL to bypass authentication or access/modify data."
    },
    {
      "issue": "Insecure Password Storage and Verification",
      "risk_level": "Critical",
      "description": "The comparison `user.password == password` strongly suggests that passwords are being stored in plaintext or a reversibly encrypted format in the database. This is a critical security flaw. Passwords must always be hashed using a strong, one-way algorithm (e

In [83]:
import json

# Get the raw output of the second task (Review Security)
raw_output_task_1 = result.tasks_output[1].raw

# Extract the JSON string by removing the markdown code block delimiters
json_string_1 = raw_output_task_1.strip().replace('```json', '').replace('```', '')

try:
    # Parse the cleaned string as a JSON dictionary
    parsed_output_1 = json.loads(json_string_1)
    print("✅ Can be parsed as JSON dictionary")
    print(f"Keys: {list(parsed_output_1.keys())}")
except json.JSONDecodeError as e:
    print(f"❌ Cannot parse as JSON: {e}")
    print("Raw string content for debugging:")
    print(json_string_1)

✅ Can be parsed as JSON dictionary
Keys: ['security_vulnerabilities', 'blocking', 'highest_risk', 'security_recommendations']


**Explanation output** The output for the Review Security task was successfully parsed as a JSON dictionary, and we can see that it contains all the expected keys: security_vulnerabilities, blocking, highest_risk, and security_recommendations. This confirms that the agents are now producing well-structured JSON outputs as intended.

### Re-run analysis when critical issues are found

## Reload Unittests Module

### Reload the `unittests` module to ensure the changes made to `unittests.py` are applied.

In [20]:
import importlib
importlib.reload(unittests)

print("unittests module reloaded successfully!")

unittests module reloaded successfully!


### Re-instantiate Tools

Re-run the cell that creates the `serper_search_tool` and `scrape_website_tool` instances to ensure they are created with the updated `unittests` definitions.

In [21]:

# create the instance of the SerperDevTool. Set the search_url to "https://owasp.org"
serper_search_tool = SerperDevTool(search_url="https://owasp.org", base_url="https://google.serper.dev/search")

# create the instance of the ScrapeWebsiteTool, which does not need any arguments
scrape_website_tool = ScrapeWebsiteTool()

### Re-run Tool Tests  

To verify that the `SerperDevTool` now runs correctly with the `search_query` argument and that both tools are functional, re-run the `unittests.test_tools` cell.


In [22]:
unittests.test_tools(serper_search_tool, scrape_website_tool)

 All tests passed!



**Exaplanatio output**  
*   The `unittests` module was successfully reloaded using `importlib.reload(unittests)`, ensuring that any recent changes to `unittests.py` were applied.
*   The `serper_search_tool` and `scrape_website_tool` instances were re-instantiated, guaranteeing they utilize the updated definitions.
*   All tests executed via `unittests.test_tools` passed successfully, confirming that the `SerperDevTool` now correctly handles the `search_query` argument and that both tools are fully functional.


## Conclusion  

This project demonstrates the effectiveness of a **multi-agent architecture** in automating code review workflows. By decomposing the process into specialized roles—code quality analysis, security evaluation, and final decision-making—the system achieves a structured and scalable approach to software validation.

The separation of concerns between agents enables:
- More accurate and focused analysis
- Improved maintainability of the system
- Clear traceability of decisions

- Comprehensive Coverage: Each agent brings unique expertise to different aspects of code review  
- Parallel Processing: Multiple agents analyze code simultaneously  
- Scalability: Easy to add new agents for additional perspectives (performance analyzer, documentation reviewer, etc.)  
- Consistency: Removes human bias from review processes  
- Efficiency: Automates time-consuming initial reviews  

Additionally, integrating external tools within the Security Engineer agent enhances the system’s ability to perform **real-world, context-aware vulnerability assessments**, going beyond static code analysis.

Overall, this architecture reflects modern software engineering practices, where **automation, modularity, and intelligent orchestration** play a critical role in maintaining code quality and security at scale.

This system serves as a strong foundation for building **automated, reliable, and extensible code review pipelines**.


## Further implementations

Experiment with different definitions for the agents and tasks. You could also test it on your own commits and see how the answers change!  

## 🚀 Future Work & Key Areas of Implementation

### 1. Feedback Loops & Iterative Refinement
Introduce feedback mechanisms between agents:
- Re-run analysis when critical issues are found
- Allow the Tech Lead to trigger re-evaluation
- Enable partial fixes and re-check cycles

---

### 2. Risk Scoring & Decision Automation
Implement a unified scoring system:
- Combine quality + security scores
- Define thresholds for automatic approval/rejection
- Example:
  - Risk < 3 → Auto-approve
  - Risk 3–7 → Request changes
  - Risk > 7 → Escalate

---

### 3. Parallel Agent Execution
Optimize performance by running:
- Quality Analysis and Security Review in parallel
- Reduce latency in large-scale systems

---

### 4. CI/CD Integration
Integrate with development pipelines:
- GitHub Actions / GitLab CI
- Automatic PR reviews
- Status checks and blocking merges

---

### 5. Memory & Context Awareness
Add persistent memory:
- Track historical issues per repository
- Learn recurring patterns (e.g., repeated security flaws)
- Improve future recommendations

---

### 6. Advanced Security Intelligence
Enhance the Security Agent with:
- CVE databases integration
- OWASP rule mapping
- Threat intelligence feeds

---

### 7. Human-in-the-Loop Review
Introduce hybrid workflows:
- Escalate complex cases to human reviewers
- Provide explainable outputs for decision transparency

---

### 8. Extensibility with Additional Agents
Expand the system with new roles:
- Performance Analyst (efficiency, scalability)
- Test Coverage Agent (unit/integration tests)
- Documentation Reviewer (code clarity)

---

### 9. Observability & Logging
Add monitoring capabilities:
- Track agent decisions and reasoning
- Log failures and inconsistencies
- Enable debugging and auditing

---

### 10. Evaluation & Benchmarking
Measure system effectiveness:
- Compare against human reviews
- Track false positives / negatives
- Continuously improve agent performance
